In [1]:
"""
BioM-BERT-Demo-Samples.ipynb

Reconstructs the exact BioM-BERT test split using the same CSV +
GroupShuffleSplit(random_state=42) used during training, runs BERT
inference on all test samples, saves to bert_demo_samples.json.
Flask reads this at demo time instead of loading the 1.2GB BERT model.

Requirements:
    - augmented_mtsamples.csv
    - exp6_aug15_gss.pkl
    - shared_tokenizer/ folder

Upload all to Colab session before running.
"""

import json
import pickle
import torch
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# Config
PKL_PATH       = "exp6_aug15_gss.pkl"
CSV_PATH       = "augmented_mtsamples.csv"
TOKENIZER_PATH = "shared_tokenizer/"
OUTPUT_PATH    = "bert_demo_samples.json"
RANDOM_STATE   = 42
BATCH_SIZE     = 16

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Load pkl
print("Loading pkl...")
with open(PKL_PATH, "rb") as f:
    data = pickle.load(f)

model_name    = data["model_name"]
num_classes   = data["num_classes"]
max_length    = data["max_length"]
front         = data["word_limit_front"]
back          = data["word_limit_back"]
label_encoder = data["label_encoder"]

config = AutoConfig.from_pretrained(model_name, num_labels=num_classes)
model  = AutoModelForSequenceClassification.from_config(config)
model.load_state_dict(data["model_state_dict"])
model.to(DEVICE)
model.eval()
print("Model loaded.")

# Load tokenizer
import os
if os.path.exists(TOKENIZER_PATH):
    print(f"Loading tokenizer from local folder: {TOKENIZER_PATH}")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
else:
    print(f"Local tokenizer not found, loading from HuggingFace: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

# Reconstruct test split (identical to training)
print("Reconstructing test split...")
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["transcription", "medical_specialty"]).reset_index(drop=True)

# Filter to the 15 specialties used in exp6
valid = set(label_encoder.classes_)
df = df[df["medical_specialty"].isin(valid)].reset_index(drop=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
_, test_idx = next(gss.split(df, groups=df["transcription"]))
test_df = df.iloc[test_idx].reset_index(drop=True)

print(f"Test set size: {len(test_df)} samples across {test_df['medical_specialty'].nunique()} specialties")

# Smart truncation
def smart_truncate(text, f=175, b=175):
    words = str(text).split()
    if len(words) <= f + b:
        return " ".join(words)
    return " ".join(words[:f] + words[-b:])

# Inference
print("Running BioM-BERT inference on test set...")

texts = test_df["transcription"].tolist()
truths = test_df["medical_specialty"].tolist()
all_preds = []

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i : i + BATCH_SIZE]
    truncated = [smart_truncate(t, front, back) for t in batch]
    enc = tokenizer(
        truncated,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        preds  = torch.argmax(logits, dim=1).cpu().numpy()
    all_preds.extend(preds)

    if (i // BATCH_SIZE) % 10 == 0:
        print(f"  {i+len(batch)}/{len(texts)} done...")

pred_labels = label_encoder.inverse_transform(all_preds)

# Build output
print("Building JSON...")
samples = []
for text, truth, pred in zip(texts, truths, pred_labels):
    words = str(text).split()
    preview = " ".join(words[:80]) + ("..." if len(words) > 80 else "")
    samples.append({
        "transcription":         text,
        "transcription_preview": preview,
        "word_count":            len(words),
        "ground_truth":          truth,
        "bert_pred":             pred,
        "bert_correct":          (pred == truth)
    })

with open(OUTPUT_PATH, "w") as f:
    json.dump(samples, f, indent=2)

Device: cuda
Loading pkl...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

Model loaded.
Loading tokenizer from local folder: shared_tokenizer/
Reconstructing test split...
Test set size: 1097 samples across 15 specialties
Running BioM-BERT inference on test set...
  16/1097 done...
  176/1097 done...
  336/1097 done...
  496/1097 done...
  656/1097 done...
  816/1097 done...
  976/1097 done...
Building JSON...


In [3]:
# Summary
correct = sum(1 for s in samples if s["bert_correct"])
print(f"\nDone. Saved {len(samples)} samples to {OUTPUT_PATH}")
print(f"BERT accuracy on saved samples: {correct/len(samples)*100:.2f}%")
print(f"(Should match Exp 6 result: ~81.49%)")


Done. Saved 1097 samples to bert_demo_samples.json
BERT accuracy on saved samples: 81.49%
(Should match Exp 6 result: ~81.49%)
